# Mesh Scene: Ray Tracing → Radar → Point Cloud

**Pipeline:**
1. Scene holds procedural meshes (wall + boxes) and lighting
2. Renderer ray-traces the scene → reflection points + intensities
3. Radar (Dirichlet solver) generates a full MIMO frame
4. Signal processing extracts a 3D point cloud

**Requires:** mitsuba (cuda_ad_rgb variant)

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[0]))

import torch
import matplotlib.pyplot as plt
from witwin.radar import Radar, Renderer, Scene
from witwin.radar.sigproc import process_pc, process_rd
from witwin.core import Box


## 1. Radar Config (TI IWR1843)

In [ ]:
config = {
    "num_tx": 3, "num_rx": 4,
    "fc": 77e9,
    "slope": 60.012,
    "adc_samples": 256,
    "adc_start_time": 6,
    "sample_rate": 4400,
    "idle_time": 7,
    "ramp_end_time": 65,
    "chirp_per_frame": 128,
    "frame_per_second": 10,
    "num_doppler_bins": 128,
    "num_range_bins": 256,
    "num_angle_bins": 64,
    "power": 15,
    "tx_loc": [[0, 0, 0], [4, 0, 0], [2, 1, 0]],
    "rx_loc": [[-6, 0, 0], [-5, 0, 0], [-4, 0, 0], [-3, 0, 0]],
}
radar = Radar(config)

## 2. Scene: Wall + Two Boxes

In [ ]:
scene = Scene()
scene.set_sensor(origin=(0, 0, 0), target=(0, 0, -5), fov=60)

wall_v, wall_f = Box(position=(0, 0, -5), size=(6, 4, 0.01)).to_mesh()
scene.add_mesh(name="wall", vertices=wall_v, faces=wall_f)

box_v, box_f = Box(position=(0.5, 0, -3), size=(0.8, 0.8, 0.8)).to_mesh()
scene.add_mesh(name="box_a", vertices=box_v, faces=box_f)

box2_v, box2_f = Box(position=(-1.0, -0.5, -4), size=(0.6, 1.0, 0.6)).to_mesh()
scene.add_mesh(name="box_b", vertices=box2_v, faces=box2_f)

renderer = Renderer(scene, resolution=128)


## 3. Ray Trace

In [ ]:
points, intensities = renderer.trace()
print(f"{points.shape[0]} reflection points")

if points.shape[0] > 0:
    print(f"X: {points[:, 0].min():.2f} ~ {points[:, 0].max():.2f} m")
    print(f"Y: {points[:, 1].min():.2f} ~ {points[:, 1].max():.2f} m")
    print(f"Z: {points[:, 2].min():.2f} ~ {points[:, 2].max():.2f} m")

## 4. Radar MIMO Frame

In [ ]:
velocity = torch.tensor([0, 0, 0.005], device='cuda')

def location_function(t):
    return intensities, points + velocity * t

frame = radar.mimo(location_function, t0=0)
print(f"Frame shape: {frame.shape}  (TX, RX, chirps, ADC)")

## 5. Point Cloud + Range-Doppler

In [ ]:
pc = process_pc(radar, frame)
print(f"{pc.shape[0]} detected points")

rd_mag, rd_map, ranges, velocities = process_rd(radar, frame, tx=0, rx=0)

## 6. Visualization

In [ ]:
fig = plt.figure(figsize=(15, 5))

# (a) Ray-traced reflection points
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
if points.shape[0] > 0:
    pts = points.cpu().numpy()
    ax1.scatter(pts[:, 0], -pts[:, 2], pts[:, 1],
                c=intensities.cpu().numpy(), cmap='hot', s=1, alpha=0.5)
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Depth (m)')
ax1.set_zlabel('Y (m)')
ax1.set_title(f'Ray-traced ({points.shape[0]} pts)')
ax1.view_init(elev=25, azim=-60)

# (b) Range-Doppler map
ax2 = fig.add_subplot(1, 3, 2)
ax2.imshow(
    rd_mag[:, :len(ranges)],
    extent=[ranges[0], ranges[-1], velocities[0], velocities[-1]],
    aspect='auto', origin='lower', cmap='jet',
)
ax2.set_xlabel('Range (m)')
ax2.set_ylabel('Velocity (m/s)')
ax2.set_title('Range-Doppler Map')

# (c) Radar point cloud
ax3 = fig.add_subplot(1, 3, 3, projection='3d')
if pc.shape[0] > 0:
    ax3.scatter(-pc[:, 0], pc[:, 1], pc[:, 2], c=pc[:, 4], cmap='hot', s=8)
    ax3.set_xlim(-4, 4)
    ax3.set_ylim(0, 8)
    ax3.set_zlim(-4, 4)
ax3.set_xlabel('X (m)')
ax3.set_ylabel('Depth (m)')
ax3.set_zlabel('Y (m)')
ax3.set_title(f'Radar point cloud ({pc.shape[0]} pts)')
ax3.view_init(elev=25, azim=-60)

plt.tight_layout()